# Práctica Usos de IA
Automatización de respuestas a emails de devolución — **Componentes Intergalácticos Industriales S.A.**

Versión adaptada para usar **Groq** (`openai/gpt-oss-120b`) en lugar de OpenAI.
Mantiene la estructura original con `LLMChain`.

## 1. Instalación de dependencias

In [19]:
%pip install langchain langchain-classic langchain-groq python-dotenv -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\macdu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Imports y configuración

In [20]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import ast

load_dotenv()

# Opción A (recomendado): crea un archivo .env con: GROQ_API_KEY=gsk_tu-clave-aqui
# Opción B: descomenta la siguiente línea
# os.environ['GROQ_API_KEY'] = 'gsk_tu-clave-aqui'

llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0)


## 3. Funciones auxiliares

In [21]:
def crear_cadena(llm, input_variables, template):
    prompt = PromptTemplate(input_variables=input_variables, template=template)
    return LLMChain(llm=llm, prompt=prompt)

def convertir_a_diccionario(texto_llm):
    import re
    # Limpiar bloques markdown
    texto_llm = re.sub(r'```python', '', texto_llm)
    texto_llm = re.sub(r'```', '', texto_llm).strip()
    # Extraer solo la parte del diccionario { ... }
    match = re.search(r'\{[^{}]+\}', texto_llm, re.DOTALL)
    if match:
        try:
            return ast.literal_eval(match.group())
        except Exception as e:
            print('Error al parsear el diccionario:', e)
            print('Fragmento encontrado:', match.group())
            exit()
    else:
        print('No se encontró ningún diccionario en la respuesta')
        print('Contenido recibido:', texto_llm)
        exit()


## 4. Paso 1 — Extraer información del email

In [22]:
extract_chain = crear_cadena(llm, ["email"],
"""A partir del siguiente correo extrae esta información:
- Número de pedido
- Nombre del remitente
- Motivo principal de la solicitud

Email:
{email}

Devuelve ÚNICAMENTE un diccionario de Python con las claves: pedido, remitente y motivo.
Sin texto adicional, sin explicaciones, solo el diccionario.
Ejemplo: pedido=#123, remitente=Juan, motivo=defecto de fábrica
""")

# debugging
print("[debug] extract_chain variables:", extract_chain.prompt.input_variables)
print("[debug] extract_chain template:\n", extract_chain.prompt.template)


[debug] extract_chain variables: ['email']
[debug] extract_chain template:
 A partir del siguiente correo extrae esta información:
- Número de pedido
- Nombre del remitente
- Motivo principal de la solicitud

Email:
{email}

Devuelve ÚNICAMENTE un diccionario de Python con las claves: pedido, remitente y motivo.
Sin texto adicional, sin explicaciones, solo el diccionario.
Ejemplo: pedido=#123, remitente=Juan, motivo=defecto de fábrica



## 5. Paso 2 — Evaluar la solicitud

In [23]:
eval_chain = crear_cadena(llm, ["motivo"],
"""Según este motivo: "{motivo}", indica si se debe ACEPTAR o RECHAZAR la devolución.

ACEPTAR si:
- Defecto de fabricación
- Error en el suministro
- Producto incompleto desde fábrica

RECHAZAR si:
- Daños durante el transporte (si no es responsabilidad de la empresa)
- Manipulación del cliente
- Solicitud fuera de plazo

Responde ÚNICAMENTE con: ACEPTAR o RECHAZAR
""")


## 6. Paso 3 — Redactar respuesta con firma personalizada

In [24]:
response_chain = crear_cadena(llm,
    ["decision", "remitente", "pedido", "nombre", "cargo", "empresa", "contacto"],
"""Redacta una respuesta formal y empática para el cliente {remitente}, sobre el pedido {pedido}.

Si la decisión es ACEPTAR:
- Agradécele por contactar.
- Confirma que el reemplazo será procesado.
- Explica brevemente que la empresa cubrirá el fallo según la política.

Si la decisión es RECHAZAR:
- Lamenta la situación.
- Explica que no se puede aceptar la devolución según la política.
- Muestra comprensión y ofrece ayuda adicional si la necesita.

Finaliza con esta firma profesional:
{nombre}
{cargo}
{empresa}
{contacto}

Decisión: {decision}
""")


## 7. Datos de firma

In [25]:
firma_info = {
    "nombre": "María Fernández",
    "cargo": "Responsable de Atención al Cliente",
    "empresa": "Componentes Intergalácticos Industriales S.A.",
    "contacto": "contacto@cii.com"
}


## 8. Caso 1 — RECHAZADA (daños en transporte)

In [26]:
email = """Asunto: Solicitud de reemplazo por daños en transporte - Pedido #D347-STELLA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Me pongo en contacto con ustedes para comunicar una incidencia relacionada con el
pedido #D347-STELLA, correspondiente a un lote de condensadores de fluzo modelo FX-88.

Al recibir el envío, varios condensadores presentaban daños visibles. Todo indica que
la mercancía sufrió una caída durante el transporte interestelar.

Solicitamos con urgencia el reemplazo inmediato de las unidades defectuosas.

Atentamente,
Darth Márquez
"""

# Paso 1
extract_result = extract_chain.invoke({"email": email})
info = convertir_a_diccionario(extract_result["text"])
pedido = info["pedido"]
remitente = info["remitente"]
motivo = info["motivo"]

# Paso 2
decision_raw = eval_chain.invoke({"motivo": motivo})
decision = decision_raw["text"].strip()

# Paso 3
response_result = response_chain.invoke({"decision": decision, "remitente": remitente, "pedido": pedido, **firma_info})

print("Decisión:", decision)
print("\nRespuesta:\n")
print(response_result["text"])


Decisión: RECHAZAR

Respuesta:

Estimado Sr. Darth Márquez,

Lamentamos mucho la situación que ha experimentado con el pedido **D347‑STELLA**. Tras revisar detenidamente su solicitud, debemos informarle que, según la política de devoluciones de Componentes Intergalácticos Industriales S.A., no nos es posible aceptar la devolución en este caso.

Entendemos lo frustrante que puede resultar esta respuesta y queremos expresarle nuestra comprensión ante los inconvenientes ocasionados. Quedamos a su disposición para brindarle cualquier asistencia adicional que necesite, ya sea para aclarar dudas sobre la política, explorar alternativas de solución o atender cualquier otro requerimiento que tenga.

Agradecemos su comprensión y quedamos atentos a sus comentarios.

Atentamente,

María Fernández  
Responsable de Atención al Cliente  
Componentes Intergalácticos Industriales S.A.  
contacto@cii.com


## 9. Caso 2 — ACEPTADA (defecto de fabricación)

In [27]:
email_aceptar = """Asunto: Devolución por defecto de fabricación - Pedido #XZ901-LUCA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Les escribo para informarles que el pedido #XZ901-LUCA, compuesto por microprocesadores
CU-92, presenta un defecto de fábrica: varias unidades no responden a la activación básica.

Solicito formalmente la devolución o el reemplazo de las unidades defectuosas.

Atentamente,
Lucía Robles
"""

# Paso 1
extract_result2 = extract_chain.invoke({"email": email_aceptar})
info2 = convertir_a_diccionario(extract_result2["text"])
pedido2 = info2["pedido"]
remitente2 = info2["remitente"]
motivo2 = info2["motivo"]

# Paso 2
decision_raw2 = eval_chain.invoke({"motivo": motivo2})
decision2 = decision_raw2["text"].strip()

# Paso 3
response_result2 = response_chain.invoke({"decision": decision2, "remitente": remitente2, "pedido": pedido2, **firma_info})

print("Decisión:", decision2)
print("\nRespuesta:\n")
print(response_result2["text"])


Decisión: ACEPTAR

Respuesta:

Estimada Sra. Lucía Robles,

Muchas gracias por ponerse en contacto con nosotros y por informarnos sobre el inconveniente presentado con el pedido **XZ901‑LUCA**.

Nos complace comunicarle que hemos iniciado el proceso de sustitución del artículo afectado. El reemplazo será gestionado de inmediato y, de acuerdo con nuestra política de garantía, la empresa cubrirá íntegramente el fallo detectado, sin coste adicional para usted.

Quedamos a su disposición para cualquier consulta o seguimiento que necesite durante este proceso.

Atentamente,

María Fernández  
Responsable de Atención al Cliente  
Componentes Intergalácticos Industriales S.A.  
contacto@cii.com
